# Link Prediction with Adamic-Adar

This notebook uses a tiny relational e-commerce database with three tables:

- `users`
- `items`
- `purchases`

We convert observed purchases into a bipartite graph: users are connected to the items they bought.

**Task.** Predict whether two users will have a future shared interest, represented here as buying the same future item.

The main point: **not every shared neighbor is equally informative**. Sharing a rare item is usually stronger evidence than sharing a very popular item.


## 1. Imports

We use only `pandas` for tables and `networkx` for graph construction and link-prediction scores.


In [1]:
from itertools import combinations
from math import log

import networkx as nx
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## 2. Set Up the Relational Data

The `period` column in `purchases` defines a simple train/future split:

- `train`: purchases observed when computing graph scores.
- `future`: purchases used only for evaluation.

The synthetic data is designed so that:

- some users share a **niche** item in the observed period and later buy the same follow-up item;
- many users share a **popular** item, but this should not be strong evidence for a future link.


In [2]:
users = pd.DataFrame([
    {"user_id": 1, "name": "Maya",  "segment": "outdoor"},
    {"user_id": 2, "name": "Noam",  "segment": "outdoor"},
    {"user_id": 3, "name": "Dana",  "segment": "coffee"},
    {"user_id": 4, "name": "Eitan", "segment": "office"},
    {"user_id": 5, "name": "Yael",  "segment": "general"},
    {"user_id": 6, "name": "Amir",  "segment": "office"},
    {"user_id": 7, "name": "Roni",  "segment": "coffee"},
    {"user_id": 8, "name": "Shira", "segment": "general"},
])

items = pd.DataFrame([
    {"item_id": 101, "item_name": "Reusable tote",       "category": "general", "role": "popular"},
    {"item_id": 102, "item_name": "Phone charger",       "category": "general", "role": "popular"},
    {"item_id": 201, "item_name": "Climbing harness",    "category": "outdoor", "role": "niche"},
    {"item_id": 202, "item_name": "Belay device",        "category": "outdoor", "role": "future follow-up"},
    {"item_id": 301, "item_name": "Espresso beans",      "category": "coffee",  "role": "niche"},
    {"item_id": 302, "item_name": "Coffee grinder",      "category": "coffee",  "role": "future follow-up"},
    {"item_id": 401, "item_name": "Laptop stand",        "category": "office",  "role": "niche"},
    {"item_id": 402, "item_name": "Ergonomic keyboard",  "category": "office",  "role": "future follow-up"},
])

purchases = pd.DataFrame([
    # Observed purchases: used to build the graph.
    {"purchase_id": 1,  "user_id": 1, "item_id": 101, "period": "train"},
    {"purchase_id": 2,  "user_id": 1, "item_id": 201, "period": "train"},
    {"purchase_id": 3,  "user_id": 2, "item_id": 102, "period": "train"},
    {"purchase_id": 4,  "user_id": 2, "item_id": 201, "period": "train"},
    {"purchase_id": 5,  "user_id": 3, "item_id": 101, "period": "train"},
    {"purchase_id": 6,  "user_id": 3, "item_id": 301, "period": "train"},
    {"purchase_id": 7,  "user_id": 4, "item_id": 101, "period": "train"},
    {"purchase_id": 8,  "user_id": 4, "item_id": 401, "period": "train"},
    {"purchase_id": 9,  "user_id": 5, "item_id": 101, "period": "train"},
    {"purchase_id": 10, "user_id": 5, "item_id": 102, "period": "train"},
    {"purchase_id": 11, "user_id": 6, "item_id": 102, "period": "train"},
    {"purchase_id": 12, "user_id": 6, "item_id": 401, "period": "train"},
    {"purchase_id": 13, "user_id": 7, "item_id": 301, "period": "train"},
    {"purchase_id": 14, "user_id": 8, "item_id": 101, "period": "train"},

    # Future purchases: hidden during scoring, used as labels for evaluation.
    {"purchase_id": 15, "user_id": 1, "item_id": 202, "period": "future"},
    {"purchase_id": 16, "user_id": 2, "item_id": 202, "period": "future"},
    {"purchase_id": 17, "user_id": 3, "item_id": 302, "period": "future"},
    {"purchase_id": 18, "user_id": 7, "item_id": 302, "period": "future"},
    {"purchase_id": 19, "user_id": 4, "item_id": 402, "period": "future"},
    {"purchase_id": 20, "user_id": 6, "item_id": 402, "period": "future"},
])

display(users)
display(items)
display(purchases)


,user_id,name,segment
0,1,Maya,outdoor
1,2,Noam,outdoor
2,3,Dana,coffee
3,4,Eitan,office
4,5,Yael,general
5,6,Amir,office
6,7,Roni,coffee
7,8,Shira,general


,item_id,item_name,category,role
0,101,Reusable tote,general,popular
1,102,Phone charger,general,popular
2,201,Climbing harness,outdoor,niche
3,202,Belay device,outdoor,future follow-up
4,301,Espresso beans,coffee,niche
5,302,Coffee grinder,coffee,future follow-up
6,401,Laptop stand,office,niche
7,402,Ergonomic keyboard,office,future follow-up


,purchase_id,user_id,item_id,period
0,1,1,101,train
1,2,1,201,train
2,3,2,102,train
3,4,2,201,train
4,5,3,101,train
5,6,3,301,train
6,7,4,101,train
7,8,4,401,train
8,9,5,101,train
9,10,5,102,train


## 3. Convert the Tables to a Graph

We build a heterogeneous bipartite graph:

- user node: `u_<user_id>`
- item node: `i_<item_id>`
- edge: observed purchase from `purchases[period == "train"]`

Only observed purchases are used for graph features. Future purchases are hidden until evaluation.


In [3]:
G = nx.Graph()

for row in users.itertuples(index=False):
    G.add_node(
        f"u_{row.user_id}",
        node_type="user",
        user_id=row.user_id,
        name=row.name,
        segment=row.segment,
    )

for row in items.itertuples(index=False):
    G.add_node(
        f"i_{row.item_id}",
        node_type="item",
        item_id=row.item_id,
        item_name=row.item_name,
        category=row.category,
        role=row.role,
    )

train_purchases = purchases[purchases["period"] == "train"]

for row in train_purchases.itertuples(index=False):
    G.add_edge(
        f"u_{row.user_id}",
        f"i_{row.item_id}",
        edge_type="purchased",
        purchase_id=row.purchase_id,
    )

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")


Nodes: 16
Edges: 14


## 4. Why Popular Items Can Mislead

For user-user link prediction, the common neighbors are items.

A simple baseline is **Common Neighbors**:

$$CN(u,v) = |N(u) \cap N(v)|$$

This counts how many observed items the two users both bought.

Adamic-Adar uses the same common neighbors, but discounts high-degree common neighbors:

$$AA(u,v) = \sum_{w \in N(u) \cap N(v)} \frac{1}{\log(deg(w))}$$

So a shared popular item contributes less than a shared niche item.


In [4]:
item_degrees = []

for node, degree in G.degree():
    if G.nodes[node]["node_type"] == "item" and degree > 0:
        item_degrees.append({
            "item_node": node,
            "item_name": G.nodes[node]["item_name"],
            "role": G.nodes[node]["role"],
            "observed_buyers": degree,
            "adamic_adar_weight_if_shared": 1 / log(degree) if degree > 1 else float("inf"),
        })

item_degree_table = pd.DataFrame(item_degrees).sort_values("observed_buyers", ascending=False)
item_degree_table


,item_node,item_name,role,observed_buyers,adamic_adar_weight_if_shared
0,i_101,Reusable tote,popular,5,0.621335
1,i_102,Phone charger,popular,3,0.910239
2,i_201,Climbing harness,niche,2,1.442695
3,i_301,Espresso beans,niche,2,1.442695
4,i_401,Laptop stand,niche,2,1.442695


The popular items have many observed buyers, so their Adamic-Adar contribution is small. Niche items have fewer observed buyers, so sharing them is stronger evidence.


## 5. Score Candidate User-User Links

A candidate pair `(u, v)` is labeled positive if the two users buy at least one same item in the future period.

The future labels are **not** used to compute the graph scores. They are used only to check whether the scores were useful.


In [5]:
user_nodes = [f"u_{user_id}" for user_id in users["user_id"]]
user_pairs = list(combinations(user_nodes, 2))

item_name_by_node = {
    f"i_{row.item_id}": row.item_name
    for row in items.itertuples(index=False)
}
user_name_by_node = {
    f"u_{row.user_id}": row.name
    for row in users.itertuples(index=False)
}

future_purchases = purchases[purchases["period"] == "future"]
future_items_by_user = (
    future_purchases
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)
item_name_by_id = dict(zip(items["item_id"], items["item_name"]))

def future_shared_items(user_node_a, user_node_b):
    user_a = int(user_node_a.split("_")[1])
    user_b = int(user_node_b.split("_")[1])
    shared_ids = future_items_by_user.get(user_a, set()) & future_items_by_user.get(user_b, set())
    return [item_name_by_id[item_id] for item_id in sorted(shared_ids)]

jaccard_scores = {
    (u, v): score
    for u, v, score in nx.jaccard_coefficient(G, user_pairs)
}

adamic_adar_scores = {
    (u, v): score
    for u, v, score in nx.adamic_adar_index(G, user_pairs)
}

rows = []

for u, v in user_pairs:
    common_item_nodes = sorted(nx.common_neighbors(G, u, v))
    common_item_names = [item_name_by_node[item_node] for item_node in common_item_nodes]
    future_items = future_shared_items(u, v)

    rows.append({
        "user_pair": f"{user_name_by_node[u]} - {user_name_by_node[v]}",
        "shared_train_items": ", ".join(common_item_names) if common_item_names else "-",
        "future_shared_items": ", ".join(future_items) if future_items else "-",
        "future_link": bool(future_items),
        "common_neighbors": len(common_item_nodes),
        "jaccard": jaccard_scores[(u, v)],
        "adamic_adar": adamic_adar_scores[(u, v)],
    })

scores = pd.DataFrame(rows)
scores = scores.sort_values(["adamic_adar", "common_neighbors"], ascending=False).reset_index(drop=True)

scores


,user_pair,shared_train_items,future_shared_items,future_link,common_neighbors,jaccard,adamic_adar
0,Maya - Noam,Climbing harness,Belay device,True,1,0.333333,1.442695
1,Dana - Roni,Espresso beans,Coffee grinder,True,1,0.500000,1.442695
2,Eitan - Amir,Laptop stand,Ergonomic keyboard,True,1,0.333333,1.442695
3,Noam - Yael,Phone charger,-,False,1,0.333333,0.910239
4,Noam - Amir,Phone charger,-,False,1,0.333333,0.910239
5,Yael - Amir,Phone charger,-,False,1,0.333333,0.910239
6,Maya - Dana,Reusable tote,-,False,1,0.333333,0.621335
7,Maya - Eitan,Reusable tote,-,False,1,0.333333,0.621335
8,Maya - Yael,Reusable tote,-,False,1,0.333333,0.621335
9,Maya - Shira,Reusable tote,-,False,1,0.500000,0.621335


## 6. Turn Scores into Predictions

For a minimal demonstration, use two simple decision rules:

- **Common-neighbors baseline:** predict a future link when `common_neighbors >= 1`.
- **Adamic-Adar:** predict a future link when `adamic_adar >= 1.0`.

In this data, `1.0` separates a shared niche item bought by 2 users from popular items bought by 3 or more users.


In [6]:
predictions = scores.copy()
predictions["pred_common_neighbors"] = predictions["common_neighbors"] >= 1
predictions["pred_adamic_adar"] = predictions["adamic_adar"] >= 1.0

predictions[[
    "user_pair",
    "shared_train_items",
    "future_shared_items",
    "future_link",
    "common_neighbors",
    "adamic_adar",
    "pred_common_neighbors",
    "pred_adamic_adar",
]]


,user_pair,shared_train_items,future_shared_items,future_link,common_neighbors,adamic_adar,pred_common_neighbors,pred_adamic_adar
0,Maya - Noam,Climbing harness,Belay device,True,1,1.442695,True,True
1,Dana - Roni,Espresso beans,Coffee grinder,True,1,1.442695,True,True
2,Eitan - Amir,Laptop stand,Ergonomic keyboard,True,1,1.442695,True,True
3,Noam - Yael,Phone charger,-,False,1,0.910239,True,False
4,Noam - Amir,Phone charger,-,False,1,0.910239,True,False
5,Yael - Amir,Phone charger,-,False,1,0.910239,True,False
6,Maya - Dana,Reusable tote,-,False,1,0.621335,True,False
7,Maya - Eitan,Reusable tote,-,False,1,0.621335,True,False
8,Maya - Yael,Reusable tote,-,False,1,0.621335,True,False
9,Maya - Shira,Reusable tote,-,False,1,0.621335,True,False


The common-neighbors baseline treats all one-item overlaps equally. For example:

- `Maya - Noam` share `Climbing harness`, a niche item, and later both buy `Belay device`.
- `Maya - Dana` share `Reusable tote`, a popular item, but no future shared item appears.

Both pairs have `common_neighbors = 1`. Adamic-Adar gives the niche overlap a much higher score.


In [7]:
def evaluate(binary_predictions, labels):
    binary_predictions = pd.Series(binary_predictions).astype(bool)
    labels = pd.Series(labels).astype(bool)

    tp = int((binary_predictions & labels).sum())
    fp = int((binary_predictions & ~labels).sum())
    fn = int((~binary_predictions & labels).sum())
    tn = int((~binary_predictions & ~labels).sum())

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "accuracy": (tp + tn) / len(labels),
    }

evaluation = pd.DataFrame([
    {"method": "Common neighbors >= 1", **evaluate(predictions["pred_common_neighbors"], predictions["future_link"])},
    {"method": "Adamic-Adar >= 1.0", **evaluate(predictions["pred_adamic_adar"], predictions["future_link"])},
])

evaluation


,method,tp,fp,fn,tn,precision,recall,accuracy
0,Common neighbors >= 1,3,13,0,12,0.1875,1.0,0.535714
1,Adamic-Adar >= 1.0,3,0,0,25,1.0000,1.0,1.000000


## Takeaway

Common neighbors asks: **Do these users share any observed item?**

Adamic-Adar asks a sharper question: **Do these users share an informative observed item?**

That distinction matters in relational graphs. Popular items create many weak overlaps. Adamic-Adar downweights those overlaps and highlights rarer shared neighbors, which are more useful for link prediction in this example.
